# 05 · Machine Learning — predecir qué líneas se devolverán

**Pregunta que se contesta aquí:** de las líneas de pedido nuevas, ¿cuáles se van a devolver?

## Por qué hace falta ML en este punto

Gold ya contesta preguntas sobre lo que pasó: qué categorías y tallas se devuelven más, cómo evoluciona la tasa por semana y qué clientes devuelven más. Pero todas esas respuestas son **promedios**. Cuando alguien hace un pedido, la tienda quiere saber algo distinto: *¿esta línea concreta, con esta talla, este descuento y este historial de cliente, se va a devolver?* Combinar muchas variables a la vez para decidir **por línea y antes de que ocurra** es lo que una tabla agregada no puede hacer, y es el momento en que entra el modelo. Con esa predicción se puede, por ejemplo, avisar al cliente de una talla que suele quedar pequeña o sugerirle otra.

## Qué pasa en este notebook

1. Se leen las features de `gold.features_linea_devolucion` y se codifican las columnas categóricas.
2. **División temporal:** el 80 % más antiguo es entrenamiento y el 20 % más reciente es prueba. Es más honesta que un reparto al azar, porque en la vida real el modelo predice el futuro a partir del pasado.
3. Se entrenan **dos modelos** y cada uno queda en su propio *run* de MLflow con parámetros, métricas y el modelo guardado:
   - una **regresión logística** (base: simple y explicable);
   - un **gradient boosting** (ensamble de árboles, que captura interacciones como "talla extrema en un vestido con descuento alto").
4. Las métricas se comparan y también se **guardan en una tabla Delta** (`gold.metricas_modelos`), para no depender solo de la interfaz de MLflow.
5. El mejor modelo se **registra en Unity Catalog** con el alias `champion`.

## Por qué no se usa la exactitud (accuracy)

Si cerca del 19 % de las líneas se devuelven, un modelo que **nunca** predice devolución acierta el 81 % y no sirve para nada. Por eso se reportan métricas centradas en la clase que importa: **recall** (de las devoluciones reales, cuántas detecta), **precision** (de las que marca, cuántas lo eran), **F1** (el equilibrio entre ambas) y **PR-AUC** (resume precision y recall en todos los umbrales y es más informativa que ROC-AUC con clases desbalanceadas).

**En esta celda** se hereda la configuración compartida.

In [0]:
%run ./00_Setup/00_Configuracion

**En esta celda** se importan MLflow, pandas y las piezas de scikit-learn que se usan (modelos, escalado, pesos por clase y métricas).

In [0]:
import mlflow
import pandas as pd
from mlflow import MlflowClient
from pyspark.sql import functions as F
from mlflow.models import infer_signature
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

**En esta celda** se leen las features desde Gold y se pasan a pandas (el tamaño lo permite). Se ordenan por fecha, porque la división de entrenamiento y prueba va a ser temporal.

In [0]:
df_features = spark.table(tbl(SCHEMA_GOLD, "features_linea_devolucion"))
pdf = df_features.toPandas()
pdf["fecha"] = pd.to_datetime(pdf["fecha"])
pdf = pdf.sort_values("fecha").reset_index(drop=True)

print(f"Filas: {len(pdf):,}")
print(f"Tasa de devolucion (fue_devuelto = 1): {pdf['fue_devuelto'].mean() * 100:.1f}%")

**En esta celda** se preparan los datos: se quitan los identificadores y la fecha (no son variables predictivas), se codifican las categorías con `get_dummies` (una columna 0/1 por categoría) y se hace la **división temporal** 80/20. Se imprime la tasa de devolución en cada parte y la exactitud de un "modelo que nunca predice devolución", que es el número que justifica no usar accuracy.

In [0]:
COLUMNAS_CATEGORICAS = ["categoria", "talla", "color", "segmento"]

pdf_encoded = pd.get_dummies(
    pdf.drop(columns=["linea_id", "cliente_id", "fecha"]),
    columns=COLUMNAS_CATEGORICAS, drop_first=True, dtype=int,
)
X = pdf_encoded.drop(columns=["fue_devuelto"])
y = pdf_encoded["fue_devuelto"].astype(int)

# Division temporal: lo mas antiguo entrena, lo mas reciente evalua (pdf ya esta ordenado por fecha)
n_train = int(len(pdf_encoded) * 0.8)
X_train, X_test = X.iloc[:n_train], X.iloc[n_train:]
y_train, y_test = y.iloc[:n_train], y.iloc[n_train:]

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}  |  Variables: {X.shape[1]}")
print(f"Tasa de devolucion en train: {y_train.mean() * 100:.1f}%  |  en test: {y_test.mean() * 100:.1f}%")
print(f"Exactitud de un modelo que NUNCA predice devolucion (test): {(1 - y_test.mean()) * 100:.1f}%  <- por eso no se usa accuracy")

**En esta celda** se configura MLflow (el registro de modelos en Unity Catalog y el experimento) y se define `entrenar_y_registrar_run()`, la función que entrena un modelo dentro de un *run*, calcula las cinco métricas sobre el conjunto de prueba y registra todo: parámetros, métricas y el modelo con su **firma** (el esquema de entradas y salidas, que Unity Catalog exige para registrar un modelo). Es el mismo patrón DRY de `escribir_delta()`: se escribe una vez y sirve para cualquier modelo, de modo que **todo modelo que se entrene deja sus métricas guardadas**.

In [0]:
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Shared/{ENTORNO}_devoluciones_moda")


def entrenar_y_registrar_run(nombre, modelo, params, fit_kwargs=None):
    """Entrena `modelo` en un run de MLflow y registra parametros, metricas y el modelo. Devuelve (fila_de_metricas, modelo)."""
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.log_params(params)
        modelo.fit(X_train, y_train, **(fit_kwargs or {}))

        proba = modelo.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)
        metricas = {
            "precision_devolucion": float(precision_score(y_test, pred, zero_division=0)),
            "recall_devolucion":    float(recall_score(y_test, pred)),
            "f1_devolucion":        float(f1_score(y_test, pred)),
            "pr_auc":               float(average_precision_score(y_test, proba)),
            "roc_auc":              float(roc_auc_score(y_test, proba)),
        }
        mlflow.log_metrics(metricas)

        firma = infer_signature(X_train.head(5), modelo.predict_proba(X_train.head(5))[:, 1])
        mlflow.sklearn.log_model(modelo, artifact_path="modelo", signature=firma, input_example=X_train.head(5))
        fila = {"modelo": nombre, "run_id": run.info.run_id, **metricas}
    return fila, modelo

**En esta celda** se entrenan los dos modelos. La regresión logística usa `class_weight="balanced"` (da más peso a la clase minoritaria) y un `StandardScaler`, porque este modelo es sensible a la escala de las variables. El gradient boosting no admite `class_weight`, así que se le pasan **pesos por muestra** equivalentes con `compute_sample_weight("balanced", ...)`. Ambos quedan en el mismo experimento y se pueden comparar.

In [0]:
resultados = []

fila_lr, modelo_lr = entrenar_y_registrar_run(
    "regresion_logistica_base",
    Pipeline([("escalado", StandardScaler()),
              ("clasificador", LogisticRegression(class_weight="balanced", max_iter=1000))]),
    params={"algoritmo": "LogisticRegression", "class_weight": "balanced", "max_iter": 1000, "escalado": "StandardScaler",
            "division": "temporal_80_20"},
)
resultados.append(fila_lr)

fila_gb, modelo_gb = entrenar_y_registrar_run(
    "gradient_boosting",
    GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=SEED),
    params={"algoritmo": "GradientBoostingClassifier", "n_estimators": 200, "max_depth": 3, "learning_rate": 0.05,
            "subsample": 0.8, "pesos": "balanced_por_muestra", "division": "temporal_80_20"},
    fit_kwargs={"sample_weight": compute_sample_weight("balanced", y_train)},
)
resultados.append(fila_gb)

df_resultados = pd.DataFrame(resultados)
display(df_resultados)

**En esta celda** se miran las diez variables que más pesan en el gradient boosting. No es una explicación causal, pero ayuda a contar la historia de negocio: si las tallas extremas, los vestidos o los descuentos altos aparecen arriba, el modelo está aprendiendo lo que sugieren las tablas de Gold.

In [0]:
importancias = pd.Series(modelo_gb.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importancias.head(10).rename("importancia").reset_index().rename(columns={"index": "variable"}))

**En esta celda** se guardan las métricas de ambos modelos en una tabla Delta, `gold.metricas_modelos`, con la fecha de entrenamiento. Se escribe en modo `append`, así que cada vez que se entrene un modelo nuevo se **acumula** su fila y se conserva el historial de todos los modelos, no solo del último.

In [0]:
df_metricas = df_resultados.assign(entrenado_en=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))
(
    spark.createDataFrame(df_metricas)
    .write.format("delta").mode("append")
    .saveAsTable(tbl(SCHEMA_GOLD, "metricas_modelos"))
)
print(f"[OK] Metricas guardadas en {tbl(SCHEMA_GOLD, 'metricas_modelos')}")
display(spark.table(tbl(SCHEMA_GOLD, "metricas_modelos")).orderBy(F.desc("entrenado_en")))

**En esta celda** se registra en Unity Catalog el modelo con mejor **PR-AUC** (la métrica más informativa con clases desbalanceadas) y se le asigna el alias `champion`. El registro crea la **versión 1** con nombre de tres niveles (`catalogo.gold.modelo_devoluciones`), con permisos y linaje como cualquier otro objeto del catálogo. Registrar no despliega el modelo: no responde predicciones hasta que se cree un endpoint de Model Serving, que es un paso aparte.

In [0]:
MODEL_NAME = tbl(SCHEMA_GOLD, "modelo_devoluciones")

mejor = df_resultados.sort_values("pr_auc", ascending=False).iloc[0]
print(f"Mejor modelo por PR-AUC: {mejor['modelo']}  (PR-AUC = {mejor['pr_auc']:.3f})")

modelo_registrado = mlflow.register_model(model_uri=f"runs:/{mejor['run_id']}/modelo", name=MODEL_NAME)

client = MlflowClient()
client.set_registered_model_alias(name=MODEL_NAME, alias="champion", version=int(modelo_registrado.version))
print(f"[OK] Modelo registrado: {MODEL_NAME}  (version {modelo_registrado.version}, alias 'champion')")

**En esta celda** se verifica el registro: se listan las versiones del modelo en Unity Catalog con su `run_id`, que enlaza cada versión con el run de MLflow donde están sus parámetros y métricas.

In [0]:
versiones = client.search_model_versions(f"name = '{MODEL_NAME}'")
print(f"Versiones registradas de {MODEL_NAME}:")
for v in versiones:
    print(f"  version {v.version}  |  run_id={v.run_id}  |  status={v.status}")

## Lectura de los resultados

- **Compara el modelo contra el punto de partida.** Un modelo útil tiene que superar a "no predecir nunca devolución" y a la tasa base de devolución. Si el PR-AUC está apenas por encima de la tasa base, el modelo casi no aporta.
- **Las métricas dependen de datos sintéticos.** La señal en los datos fue diseñada por el autor del proyecto (tallas extremas, categorías, descuentos, tipo de cliente) más ruido aleatorio, así que el valor del proyecto está en la arquitectura y las decisiones, no en la cifra de las métricas.
- **Precaución con métricas demasiado buenas.** Un PR-AUC cercano a 1 sería una señal de fuga de datos. Por eso las features de historial usan solo líneas con al menos 35 días de antigüedad y la división es temporal.
- **Siguiente paso posible (no incluido):** crear un endpoint de Model Serving para el modelo registrado y probarlo con una petición de ejemplo.

Machine Learning · MLflow y Unity Catalog &nbsp;·&nbsp; Fin del proyecto